In [12157]:
import pandas as pd
import regex as re
import io

In [12158]:
def read_vcf(path):
    with open(path, 'r') as f:
        lines = [l for l in f if not l.startswith('##')]
    return pd.read_csv(
        io.StringIO(''.join(lines)),
        dtype={'#CHROM': str, 'POS': int, 'ID': str, 'REF': str, 'ALT': str,
               'QUAL': str, 'FILTER': str, 'INFO': str},
        sep='\t'
    ).rename(columns={'#CHROM': 'CHROM'})

def extract_af(info):
    match = re.search(r'AF=([\d\.]+)', info)
    return match.group(1) if match else None

In [12159]:
df = read_vcf('/Users/fionachow/Documents/NYU/CDS/Spring 2024/Research Fair/Hochwagen/Individual Data/ERR3240115/ERR3240115_rDNA.vcf') #vcf file from original variant calling of 1 individual
df['AF'] = df['INFO'].apply(extract_af)

df[df['POS']==1671]

,CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,AF
52,Human,1671,.,GC,G,1432,PASS,"DP=7755;AF=0.031593;SB=44;DP4=1326,1605,145,10...",0.031593
53,Human,1671,.,GCTCCC,G,12978,PASS,"DP=7755;AF=0.596132;SB=1;DP4=1326,1605,2116,25...",0.596132


In [12160]:
type(df['AF'].iloc[0])

str

In [12161]:
#get sequnce of individual at each position
og_ref = pd.read_csv('/Users/fionachow/Documents/NYU/CDS/Summer 2024/Human rDNA Research/Project/Human-rDNA/outputs/src_outputs/og_ref.csv')

og_ref.shape

(13314, 3)

In [12162]:
type(og_ref['POS'].iloc[30])

numpy.int64

In [12163]:
new_ref = pd.read_csv('/Users/fionachow/Documents/NYU/CDS/Summer 2024/Human rDNA Research/Project/Human-rDNA/outputs/src_outputs/1000_genome_new_ref/1000_genome_new_ref_v2.csv')

new_ref.shape

(13314, 3)

In [12164]:
df = df[['POS', 'REF', 'ALT', 'AF']]  

print(og_ref.columns)
print(df.columns)

merged_df = pd.merge(og_ref, df, on='POS', how='left', suffixes=('_og', '_vcf'))

merged_df.rename(columns={'REF_og':'ALT_og', 'ALT':'ALT_vcf'}, inplace=True)

merged_df.head(33)

Index(['POS', 'Type', 'REF'], dtype='object')
Index(['POS', 'REF', 'ALT', 'AF'], dtype='object')


,POS,Type,ALT_og,REF_vcf,ALT_vcf,AF
0,1,sequence,G,NaN,NaN,NaN
1,2,sequence,C,NaN,NaN,NaN
2,3,sequence,T,NaN,NaN,NaN
3,4,sequence,G,NaN,NaN,NaN
4,5,sequence,A,NaN,NaN,NaN
5,6,sequence,C,NaN,NaN,NaN
6,7,sequence,A,NaN,NaN,NaN
7,8,sequence,C,NaN,NaN,NaN
8,9,sequence,G,NaN,NaN,NaN
9,10,sequence,C,NaN,NaN,NaN


In [12165]:
merged_df.shape

(13346, 6)

In [12166]:
# Combine the ALT_og column and ALT_vcf into ALT_combined, and set AF to 0 where variant did not exist
merged_df['ALT_combined'] = merged_df.apply(lambda x: x['ALT_vcf'] if pd.notna(x['ALT_vcf']) else x['ALT_og'], axis=1)
merged_df['AF'] = merged_df['AF'].fillna(0) 

merged_df.head(32)

,POS,Type,ALT_og,REF_vcf,ALT_vcf,AF,ALT_combined
0,1,sequence,G,NaN,NaN,0,G
1,2,sequence,C,NaN,NaN,0,C
2,3,sequence,T,NaN,NaN,0,T
3,4,sequence,G,NaN,NaN,0,G
4,5,sequence,A,NaN,NaN,0,A
5,6,sequence,C,NaN,NaN,0,C
6,7,sequence,A,NaN,NaN,0,A
7,8,sequence,C,NaN,NaN,0,C
8,9,sequence,G,NaN,NaN,0,G
9,10,sequence,C,NaN,NaN,0,C


In [12167]:
merged_df.drop(columns=['ALT_vcf','Type'], inplace=True)

merged_df.rename(columns={'ALT_og': 'og_ref', 'AF':'AF_old', 'ALT_combined':'ALT_old'}, inplace=True)

merged_df.head(73)

,POS,og_ref,REF_vcf,AF_old,ALT_old
0,1,G,NaN,0,G
1,2,C,NaN,0,C
2,3,T,NaN,0,T
3,4,G,NaN,0,G
4,5,A,NaN,0,A
...,...,...,...,...,...
68,69,G,NaN,0,G
69,70,C,NaN,0,C
70,71,C,NaN,0,C
71,72,T,NaN,0,T


In [12168]:
type(merged_df['AF_old'].iloc[30])

str

In [12169]:
merged_df = pd.merge(merged_df, new_ref, on='POS', how='left')
merged_df.drop(columns=['Unnamed: 0'], inplace=True)
merged_df.rename(columns={'1000_genome_new_ref': 'new_ref'}, inplace=True)

merged_df

,POS,og_ref,REF_vcf,AF_old,ALT_old,new_ref
0,1,G,NaN,0,G,G
1,2,C,NaN,0,C,C
2,3,T,NaN,0,T,T
3,4,G,NaN,0,G,G
4,5,A,NaN,0,A,A
...,...,...,...,...,...,...
13341,13310,T,NaN,0,T,T
13342,13311,C,NaN,0,C,C
13343,13312,G,NaN,0,G,G
13344,13313,C,NaN,0,C,C


In [12170]:
merged_df[(merged_df['new_ref'].str.len()) > (merged_df['og_ref'].str.len())].shape

(48, 6)

In [12171]:
filtered_df = merged_df[(merged_df['og_ref'] != merged_df['new_ref']) & (merged_df['AF_old'] != 0)]

filtered_df.shape #number of ALT rows being changed

(87, 6)

In [12172]:
#using ALT_old unless reference changed and AF_old != 0
merged_df['ALT_new'] = merged_df.apply(lambda x: x['og_ref'] if (x['AF_old'] != 0) and (x['og_ref'] != x['new_ref']) else x['ALT_old'], axis=1)

merged_df

,POS,og_ref,REF_vcf,AF_old,ALT_old,new_ref,ALT_new
0,1,G,NaN,0,G,G,G
1,2,C,NaN,0,C,C,C
2,3,T,NaN,0,T,T,T
3,4,G,NaN,0,G,G,G
4,5,A,NaN,0,A,A,A
...,...,...,...,...,...,...,...
13341,13310,T,NaN,0,T,T,T
13342,13311,C,NaN,0,C,C,C
13343,13312,G,NaN,0,G,G,G
13344,13313,C,NaN,0,C,C,C


In [12173]:
filtered_df2 = merged_df[(merged_df['og_ref'] != merged_df['new_ref'])]

filtered_df2.shape #number of AF rows being changed

(110, 7)

In [12174]:
#using AF_old unless reference changed 
merged_df['AF_new'] = merged_df.apply(lambda x: 1 - float(x['AF_old']) if (x['og_ref'] != x['new_ref']) else float(x['AF_old']), axis=1) #note: 'AF_new' is all in float, 'AF_old' was in string and 0 was integer

merged_df

,POS,og_ref,REF_vcf,AF_old,ALT_old,new_ref,ALT_new,AF_new
0,1,G,NaN,0,G,G,G,0.0
1,2,C,NaN,0,C,C,C,0.0
2,3,T,NaN,0,T,T,T,0.0
3,4,G,NaN,0,G,G,G,0.0
4,5,A,NaN,0,A,A,A,0.0
...,...,...,...,...,...,...,...,...
13341,13310,T,NaN,0,T,T,T,0.0
13342,13311,C,NaN,0,C,C,C,0.0
13343,13312,G,NaN,0,G,G,G,0.0
13344,13313,C,NaN,0,C,C,C,0.0


In [12175]:
#deal w deletions here
merged_df[merged_df['REF_vcf'].str.len() > 1]

,POS,og_ref,REF_vcf,AF_old,ALT_old,new_ref,ALT_new,AF_new
102,103,G,GC,0.987559,G,G,G,0.987559
537,537,T,TG,0.957346,T,T,T,0.957346
633,633,G,GT,0.052245,G,G,G,0.052245
763,763,C,CGG,0.649318,C,C,C,0.649318
963,963,T,TGAGAC,0.008979,T,T,T,0.008979
...,...,...,...,...,...,...,...,...
11093,11069,G,GC,0.935852,G,G,G,0.935852
11142,11118,G,GC,0.863450,G,G,G,0.863450
11292,11268,C,CG,0.718024,C,C,C,0.718024
11442,11417,G,GGCA,0.205890,G,G,G,0.205890


In [12176]:
merged_df['AF_new'] = merged_df.apply(lambda x: 0 if (x['new_ref'] == x['ALT_new']) else x['AF_new'], axis=1)

merged_df.head(104)

,POS,og_ref,REF_vcf,AF_old,ALT_old,new_ref,ALT_new,AF_new
0,1,G,NaN,0,G,G,G,0.0
1,2,C,NaN,0,C,C,C,0.0
2,3,T,NaN,0,T,T,T,0.0
3,4,G,NaN,0,G,G,G,0.0
4,5,A,NaN,0,A,A,A,0.0
...,...,...,...,...,...,...,...,...
99,100,C,NaN,0,C,C,C,0.0
100,101,C,NaN,0,C,C,C,0.0
101,102,T,NaN,0,T,T,T,0.0
102,103,G,GC,0.987559,G,G,G,0.0


In [12177]:
merged_df['AF_old'] = pd.to_numeric(merged_df['AF_old'], errors='raise')

copy_del_df = merged_df[merged_df['REF_vcf'].str.len() > 1].copy()

copy_del_df['add_len'] = copy_del_df['REF_vcf'].str.len() - 1
copy_del_df['inv_AF'] = 1 - copy_del_df['AF_old']

rows = []

for _, row in copy_del_df.iterrows():
    for i in range(1, row['add_len'] + 1):
        new_row = {
            'scenario': '>0.5' if row['AF_old'] > 0.5 else '<=0.5',
            'POS': row['POS'] + i,
            'add_len': row['add_len'],
            'AF': row['AF_old'],
            'inv_AF': row['inv_AF'],
            # 'del_POS': row['POS'] + i
        }
        rows.append(new_row)

new_df = pd.DataFrame(rows)

display(new_df.head())


,scenario,POS,add_len,AF,inv_AF
0,>0.5,104,1,0.987559,0.012441
1,>0.5,538,1,0.957346,0.042654
2,<=0.5,634,1,0.052245,0.947755
3,>0.5,764,2,0.649318,0.350682
4,>0.5,765,2,0.649318,0.350682


In [12178]:
# Merge the two DataFrames on 'POS'
merged_updated_df = pd.merge(merged_df, new_df[['scenario', 'POS', 'AF', 'inv_AF']], on='POS', how='left')


merged_updated_df.head(766)

,POS,og_ref,REF_vcf,AF_old,ALT_old,new_ref,ALT_new,AF_new,scenario,AF,inv_AF
0,1,G,NaN,0.000000,G,G,G,0.0,NaN,NaN,NaN
1,2,C,NaN,0.000000,C,C,C,0.0,NaN,NaN,NaN
2,3,T,NaN,0.000000,T,T,T,0.0,NaN,NaN,NaN
3,4,G,NaN,0.000000,G,G,G,0.0,NaN,NaN,NaN
4,5,A,NaN,0.000000,A,A,A,0.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
761,761,C,NaN,0.000000,C,C,C,0.0,NaN,NaN,NaN
762,762,T,NaN,0.000000,T,T,T,0.0,NaN,NaN,NaN
763,763,C,CGG,0.649318,C,C,C,0.0,NaN,NaN,NaN
764,764,G,NaN,0.000000,G,G,G,0.0,>0.5,0.649318,0.350682


In [12179]:
# Update 'AF_new' based on condition
merged_updated_df.loc[merged_updated_df['new_ref'] == '0', 'AF_new'] = merged_updated_df['inv_AF']

merged_updated_df.head(104)


,POS,og_ref,REF_vcf,AF_old,ALT_old,new_ref,ALT_new,AF_new,scenario,AF,inv_AF
0,1,G,NaN,0.000000,G,G,G,0.000000,NaN,NaN,NaN
1,2,C,NaN,0.000000,C,C,C,0.000000,NaN,NaN,NaN
2,3,T,NaN,0.000000,T,T,T,0.000000,NaN,NaN,NaN
3,4,G,NaN,0.000000,G,G,G,0.000000,NaN,NaN,NaN
4,5,A,NaN,0.000000,A,A,A,0.000000,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
99,100,C,NaN,0.000000,C,C,C,0.000000,NaN,NaN,NaN
100,101,C,NaN,0.000000,C,C,C,0.000000,NaN,NaN,NaN
101,102,T,NaN,0.000000,T,T,T,0.000000,NaN,NaN,NaN
102,103,G,GC,0.987559,G,G,G,0.000000,NaN,NaN,NaN


In [12180]:
merged_updated_df['ALT_new'] = merged_updated_df.apply(lambda x: '0' if pd.notna(x['inv_AF']) and x['new_ref']!='0' else x['ALT_new'], axis=1)
merged_updated_df['AF_new'] = merged_updated_df.apply(lambda x: x['AF'] if pd.notna(x['inv_AF']) and x['ALT_new']=='0' else x['AF_new'], axis=1)


In [12181]:
#dealing with insertions with duplicate rows here
temp = df[(df['REF'].str.len()) < (df['ALT'].str.len())]
duplicates = temp.duplicated(subset='POS', keep=False)
duplicate_rows = temp[duplicates]
duplicate_rows['Duplicate'] = 'Y'
duplicate_rows = duplicate_rows[['POS','Duplicate']]
unique_rows = duplicate_rows.drop_duplicates()


unique_rows

/var/folders/hb/l21bg8lx3tgc5ky3s56cwdth0000gn/T/ipykernel_69148/2392623350.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  duplicate_rows['Duplicate'] = 'Y'


,POS,Duplicate
13,386,Y
82,2489,Y
235,7512,Y
258,8688,Y
263,8706,Y
331,11402,Y
348,12047,Y
358,12700,Y
360,12722,Y
364,12969,Y


In [12182]:
# if insertion w duplicate rows in original vcf, and new_ref != ALT_old, then make ALT_new = ALT_old
# Motivation: otherwise there would be two identical rows but w different AF which would not make sense
# example pos 386
merged_updated_df = pd.merge(merged_updated_df, unique_rows, on='POS', how='left')
merged_updated_df['ALT_new'] = merged_updated_df.apply(lambda x: x['ALT_old'] if pd.notna(x['Duplicate']) and x['new_ref']!=x['ALT_old'] else x['ALT_new'] , axis=1)
merged_updated_df['AF_new'] = merged_updated_df.apply(lambda x: x['AF_old'] if pd.notna(x['Duplicate']) and x['new_ref']!=x['ALT_old'] else x['AF_new'] , axis=1)

merged_updated_df

,POS,og_ref,REF_vcf,AF_old,ALT_old,new_ref,ALT_new,AF_new,scenario,AF,inv_AF,Duplicate
0,1,G,NaN,0.0,G,G,G,0.0,NaN,NaN,NaN,NaN
1,2,C,NaN,0.0,C,C,C,0.0,NaN,NaN,NaN,NaN
2,3,T,NaN,0.0,T,T,T,0.0,NaN,NaN,NaN,NaN
3,4,G,NaN,0.0,G,G,G,0.0,NaN,NaN,NaN,NaN
4,5,A,NaN,0.0,A,A,A,0.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
13436,13310,T,NaN,0.0,T,T,T,0.0,NaN,NaN,NaN,NaN
13437,13311,C,NaN,0.0,C,C,C,0.0,NaN,NaN,NaN,NaN
13438,13312,G,NaN,0.0,G,G,G,0.0,NaN,NaN,NaN,NaN
13439,13313,C,NaN,0.0,C,C,C,0.0,NaN,NaN,NaN,NaN


<!-- merged_updated_df = pd.merge(merged_df, new_df[['scenario', 'POS', 'AF', 'inv_AF']], on='POS', how='left') -->

In [12183]:
# just for slides
merged_updated_df.drop(columns=['scenario','AF', 'inv_AF'], inplace=True)
new_order = ['POS','og_ref', 'REF_vcf', 'new_ref', 'ALT_old', 'ALT_new', 'AF_old', 'AF_new']
display(merged_updated_df.iloc[[0,30,31,72,73,79,102,103,208,385,386,749,763,764,765,1672,1673,1674,1675,1676,1677,1678,1679,1680,1681,1682,1683,1684,1755,2488,2489,2490,2491,2492,2493,2494,2495]][new_order])

,POS,og_ref,REF_vcf,new_ref,ALT_old,ALT_new,AF_old,AF_new
0,1,G,NaN,G,G,G,0.000000,0.000000
30,31,T,T,C,C,T,1.000000,0.000000
31,32,C,C,T,T,C,1.000000,0.000000
72,73,C,C,A,A,C,0.996952,0.003048
73,74,A,A,C,C,A,0.999624,0.000376
79,80,A,A,AC,AC,A,0.977024,0.022976
102,103,G,GC,G,G,G,0.987559,0.000000
103,104,C,NaN,0,C,C,0.000000,0.012441
208,209,C,C,C,G,G,0.109203,0.109203
385,386,T,T,TGGCC,TGGCAA,TGGCAA,0.100669,0.100669


In [12184]:
merged_updated_df[merged_updated_df['AF_new']==1]

,POS,og_ref,REF_vcf,AF_old,ALT_old,new_ref,ALT_new,AF_new,Duplicate
11001,10882,G,NaN,0.0,G,C,G,1.0,NaN


In [12185]:
merged_updated_df.drop(columns=['Duplicate','REF_vcf', 'og_ref', 'AF_old', 'ALT_old'], inplace=True)

display(merged_updated_df.head())

,POS,new_ref,ALT_new,AF_new
0,1,G,G,0.0
1,2,C,C,0.0
2,3,T,T,0.0
3,4,G,G,0.0
4,5,A,A,0.0


In [12186]:
merge_updated_df = merged_updated_df[merged_updated_df['AF_new']!=0] #new vcf output

In [12187]:
merge_updated_df.shape

(569, 4)

In [12188]:
merge_updated_df.head(60)

,POS,new_ref,ALT_new,AF_new
72,73,A,C,0.003048
73,74,C,A,0.000376
79,80,AC,A,0.022976
103,104,0,C,0.012441
120,121,G,C,0.001038
208,209,C,G,0.109203
236,237,GC,G,0.033079
269,270,CG,C,0.054447
274,275,CG,C,0.062054
283,284,AC,A,0.039862


In [12189]:
merge_updated_df[merge_updated_df['AF_new']>0.5].shape

(23, 4)

In [12190]:
merge_updated_df[merge_updated_df['AF_new']>0.5]

,POS,new_ref,ALT_new,AF_new
764,764,G,0,0.649318
765,765,G,0,0.649318
1674,1672,0,C,0.968407
1678,1674,0,C,0.681525
1680,1675,0,C,0.681525
1682,1676,0,C,0.681525
1755,1749,C,T,0.570535
2493,2487,G,0,0.542124
2494,2488,C,0,0.542124
5977,5892,0,G,0.843277
